# Challenge 1 (Binary): colour-histogram baseline

**Nevus vs others** using only colour histograms and a simple classifier (Logistic Regression and kNN).

- **Experiment 0**: `PREPROCESS = False`, i.e. raw images (only resized for speed)
- **Experiment 1**: `PREPROCESS = True`, i.e. the same features after hair removal

The models are trained on `train` only. The `val` split is used just for reporting (confusion matrix and metrics).
Features are cached to `Binary/features_cache/`, so re-running with the same configuration is instant.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from joblib import Parallel, delayed

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)

# make the project root importable (this notebook lives in Binary/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name in ("Binary", "Multiclass", "notebooks") else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing import load_image, remove_hair, resize_image
from feature_extraction import color_histogram, color_histogram_3d, color_moments

SEED = 42
print("Project root:", PROJECT_ROOT)

## 0. Configuration

In [ ]:
DATA_DIR = PROJECT_ROOT / "Binary"

PREPROCESS = False          # False = experiment 0 (raw), True = experiment 1 (hair removal)
SHORT_SIDE = 256            # resize for speed (normalised histograms barely change with resizing)

FEATURE = "hist"            # "hist" (per-channel), "hist3d" (joint) or "moments"
COLOR_SPACE = "hsv"         # "rgb", "hsv", "lab", "ycrcb"
BINS = 32                   # per channel for "hist"; for "hist3d" use e.g. 8 (-> 512 features)

N_JOBS = -1                 # parallel feature extraction (-1 = all cores)
MAX_TRAIN_PER_CLASS = None  # e.g. 1000 for quick model exploration (random, stratified); None = all.
                            # Validation is ALWAYS used in full so results stay comparable.

CACHE_DIR = DATA_DIR / "features_cache"
CACHE_DIR.mkdir(exist_ok=True)

## 1. Image list and labels

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def list_split(split, max_per_class=None):
    split_dir = next(p for p in DATA_DIR.iterdir() if p.is_dir() and p.name.lower().startswith(split))
    rows = []
    for cls_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        files = sorted(f for f in cls_dir.rglob("*") if f.suffix.lower() in IMG_EXT)
        if max_per_class and len(files) > max_per_class:
            rng = np.random.default_rng(SEED)
            files = sorted(rng.choice(files, max_per_class, replace=False))
        rows += [{"path": f, "class": cls_dir.name.lower()} for f in files]
    return pd.DataFrame(rows)

train_df, val_df = list_split("train", MAX_TRAIN_PER_CLASS), list_split("val")

# label: nevus = 0, others (abnormal) = 1  -> "positive" = non-nevus lesion
classes = sorted(train_df["class"].unique())
NEG = next((c for c in classes if c.startswith("nev")), classes[0])
for d in (train_df, val_df):
    d["y"] = (d["class"] != NEG).astype(int)
LABELS = [NEG, "others"]

print("classes:", classes, "| negative class (0):", NEG)
pd.concat({"train": train_df["class"].value_counts(), "val": val_df["class"].value_counts()}, axis=1)

## 2. Preprocessing check

A quick look at what the preprocessing does on a few training images (only matters when `PREPROCESS = True`).

In [ ]:
from preprocessing import hair_mask

sample = train_df.sample(4, random_state=SEED)["path"].tolist()
fig, axes = plt.subplots(3, len(sample), figsize=(3.2 * len(sample), 9))
for k, p in enumerate(sample):
    raw = resize_image(cv2.imread(str(p)), SHORT_SIDE)
    clean, mask = remove_hair(raw, return_mask=True)
    for r, (im, t) in enumerate([(raw, "raw (resized)"), (mask, "hair mask"), (clean, "hair removed")]):
        ax = axes[r, k]
        ax.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap="gray")
        ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Feature extraction (cached)

In [ ]:
def extract_one(path):
    if PREPROCESS:
        img = load_image(path, short_side=SHORT_SIDE, hair=True, color_constancy=False)
    else:
        img = resize_image(cv2.imread(str(path)), SHORT_SIDE)
    if FEATURE == "hist":
        return color_histogram(img, COLOR_SPACE, BINS)
    if FEATURE == "hist3d":
        return color_histogram_3d(img, COLOR_SPACE, BINS)
    if FEATURE == "moments":
        return color_moments(img, COLOR_SPACE)
    raise ValueError(FEATURE)

def get_features(df, split):
    tag = f"{split}_{FEATURE}_{COLOR_SPACE}_b{BINS}_s{SHORT_SIDE}_pre{int(PREPROCESS)}_n{len(df)}"
    f = CACHE_DIR / f"{tag}.npz"
    if f.exists():
        d = np.load(f, allow_pickle=True)
        if list(d["paths"]) == [str(p) for p in df["path"]]:
            print(f"[cache] {f.name}")
            return d["X"]
    t0 = time.time()
    X = np.stack(Parallel(n_jobs=N_JOBS, batch_size=64)(delayed(extract_one)(p) for p in df["path"]))
    np.savez_compressed(f, X=X, paths=np.array([str(p) for p in df["path"]]))
    print(f"[extracted] {split}: {X.shape} in {time.time() - t0:.0f}s -> {f.name}")
    return X

X_train, y_train = get_features(train_df, "train"), train_df["y"].values
X_val, y_val = get_features(val_df, "val"), val_df["y"].values
print("X_train", X_train.shape, "| X_val", X_val.shape)

In [ ]:
# Mean histogram per class: which bins separate nevus from others?
if FEATURE == "hist":
    n_ch = X_train.shape[1] // BINS
    names = {"hsv": "HSV", "rgb": "BGR", "lab": "Lab", "ycrcb": "YCrCb"}.get(COLOR_SPACE, COLOR_SPACE)
    fig, axes = plt.subplots(1, n_ch, figsize=(5 * n_ch, 3.2))
    for c in range(n_ch):
        for lbl, name, col in [(0, NEG, "#4C72B0"), (1, "others", "#DD8452")]:
            axes[c].plot(X_train[y_train == lbl, c*BINS:(c+1)*BINS].mean(0), label=name, color=col)
        axes[c].set_title(f"{COLOR_SPACE.upper()} channel {names[c] if len(names) == n_ch else c}")
        axes[c].set_xlabel("bin")
    axes[0].set_ylabel("mean frequency"); axes[0].legend(frameon=False)
    plt.tight_layout(); plt.show()

## 4. Models

- **Logistic Regression** with a small grid on `C`
- **kNN** with a small grid on `k` and the weighting

Both use 5-fold cross-validation on the **training set only**. Before scaling, the histograms get a `sqrt` (Hellinger) transform, which makes Euclidean distances behave better on distributions. `class_weight="balanced"` is not needed here because the binary set is roughly 50/50.

In [ ]:
hellinger = FunctionTransformer(np.sqrt) if FEATURE.startswith("hist") else "passthrough"
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

models = {
    "LogReg": GridSearchCV(
        make_pipeline(hellinger, StandardScaler(), LogisticRegression(max_iter=5000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10]},
        cv=cv, scoring="balanced_accuracy", n_jobs=N_JOBS),
    "kNN": GridSearchCV(
        make_pipeline(hellinger, StandardScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [5, 11, 21, 41],
         "kneighborsclassifier__weights": ["uniform", "distance"]},
        cv=cv, scoring="balanced_accuracy", n_jobs=N_JOBS),
}

results = []
for name, gs in models.items():
    t0 = time.time()
    gs.fit(X_train, y_train)
    pred = gs.predict(X_val)
    prob = gs.predict_proba(X_val)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    results.append({
        "model": name,
        "best_params": gs.best_params_,
        "cv_bal_acc (train)": round(gs.best_score_, 4),
        "val_accuracy": round(accuracy_score(y_val, pred), 4),
        "val_bal_accuracy": round(balanced_accuracy_score(y_val, pred), 4),
        "val_sensitivity": round(tp / (tp + fn), 4),
        "val_specificity": round(tn / (tn + fp), 4),
        "val_auc": round(roc_auc_score(y_val, prob), 4),
        "fit_time_s": round(time.time() - t0, 1),
    })

res = pd.DataFrame(results).set_index("model")
res

## 5. Confusion matrices (validation)

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.2))
for ax, (name, gs) in zip(np.atleast_1d(axes), models.items()):
    ConfusionMatrixDisplay.from_predictions(y_val, gs.predict(X_val), display_labels=LABELS,
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{name}  (acc {res.loc[name, 'val_accuracy']:.3f})")
plt.tight_layout(); plt.show()

for name, gs in models.items():
    print(f"--- {name} ---")
    print(classification_report(y_val, gs.predict(X_val), target_names=LABELS, digits=3))

## 6. Save results

In [ ]:
exp_name = f"exp{int(PREPROCESS)}_{FEATURE}_{COLOR_SPACE}_b{BINS}"
out = DATA_DIR / "results_baseline.csv"
row = res.reset_index().assign(experiment=exp_name, preprocess=PREPROCESS, feature=FEATURE,
                               color_space=COLOR_SPACE, bins=BINS, n_features=X_train.shape[1])
row["best_params"] = row["best_params"].astype(str)
if out.exists():
    old = pd.read_csv(out)
    old = old[old["experiment"] != exp_name]
    row = pd.concat([old, row], ignore_index=True)
row.to_csv(out, index=False)
print("saved ->", out)
row

## 7. Next steps

1. Re-run with `PREPROCESS = True` and compare the two rows in `results_baseline.csv`.
2. Try `COLOR_SPACE` = rgb / hsv / lab and `BINS` = 16 / 32 / 64, plus `FEATURE = "hist3d"` with `BINS = 8`.
3. Then compute the histogram **inside the lesion mask** only (needs segmentation), and add texture (LBP) and shape features.